# Dirt Sideband using CC muons

We use a general selection of CC muons to extract an appropriate dirt scaling factor to apply to the dirt events for our signal and skyshine sideband.

In [1]:
print('Loading Packages...\n')
from lib.imports import *
from lib.constants import *
from lib.reco_functions import (
    reco_energy,
    bunch_cutting,
    pairwise_relative_direction,
    is_inside_rotated_parabola,
)

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
# Difference between Empirical and Base GENIE is small for CC events
# this is because we only edited the de-excitation physics
# for high-energy muons we see essentially 0 difference

# we can use Empirical GENIE to be self-consistent
POT = 4.41733e20      # MC POT

# we also re-define the bunch start and end criteria as we aren't doing bunch cutting
bunch_time_cutoff = 1560

# spill duration for data (based on fit results)
spill_start_22 = 206       # this captures the entirity of the spill (centers - 18.936/2, centers + 18.936/2)
spill_end_22 = 1739

spill_start_23 = 189
spill_end_23 = 1719

# REQUIRES USER INPUT (specify path to MC files)
MC_directory = '../MC_GENIE-O16/'        # 'MC_GENIE-Base/'
plots_dir = 'plots/sidebands/'           # directory to hold plots
os.system('mkdir -p ' + plots_dir)        # auto-generate plots directory if not present

data_directory = '../../FY22_23/'
csv_file = 'lib/POT_trigger_FY22_23_summary.csv'   # contains trigger counts and POT for on-beam data
offbeam_directory = '../../off_beam/'
csv_file_ext = 'lib/offbeam_summary.csv'           # contains trigger counts for off-beam data

print('done')

Loading Packages...

done


In [3]:
# MC

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]
WEIGHTING = True

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_PDG               = []
MC_file_index        = []   # which file each cluster came from
MC_event_number      = []
MC_MRDT = []
    
if WEIGHTING:
    MC_CV_weight = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_event_number.append(CEN)
    MC_file_index.append(np.full(len(CEN), file_idx, dtype=np.int32))

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_PDG.append(S['trueNuPDG'].array(library='np')[CEN])
    MC_MRDT.append(S['numMRDTracks'].array(library='np')[CEN])
    
    if WEIGHTING:
        MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])
        

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_event_number    = np.concatenate(MC_event_number)
MC_file_index      = np.concatenate(MC_file_index)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_NC              = np.concatenate(MC_NC)
MC_PDG             = np.concatenate(MC_PDG)
MC_MRDT            = np.concatenate(MC_MRDT)
    
if WEIGHTING:
    MC_CV_weight = np.concatenate(MC_CV_weight)


print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

Loading MC files: 100%|████████████████████| 3922/3922 [01:43<00:00, 38.05it/s]



Managing arrays...

POT: 4.417e+20
Total clusters: 3278461

done


In [4]:
# on-beam data  

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

file_names = [file for file in os.listdir(data_directory) if os.path.isfile(os.path.join(data_directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file
pot_df = pd.read_csv(csv_file)

# extract total POT and triggers
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
MRD_tracks = []
isBrightest = []
cluster_Number = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 25 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(data_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_Number.append(Event["cluster_Number"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        MRD_tracks.append(Event["MRD_Track"].array(library="np"))
        isBrightest.append(Event["isBrightest"].array(library="np"))
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_Number = np.concatenate(cluster_Number)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
MRD_tracks = np.concatenate(MRD_tracks)
isBrightest = np.concatenate(isBrightest)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

There are:  389  files

( 0 / 389 )
( 25 / 389 )
( 50 / 389 )
( 75 / 389 )
( 100 / 389 )
( 125 / 389 )
( 150 / 389 )
( 175 / 389 )
( 200 / 389 )
( 225 / 389 )
( 250 / 389 )
( 275 / 389 )
( 300 / 389 )
( 325 / 389 )
( 350 / 389 )
( 375 / 389 )

Managing arrays...

60624559 triggers
2.572970591694 e20 POT



In [5]:
# off beam data

# The off-beam triggers in ANNIE DO NOT record MRD event information - it's OFF
# this is a flaw in all ANNIE analyses and appropriate systematics need to be applied
# Therefore we rely on the pre-beam on-beam data as our "off-beam" sample since

# pre-beam (pre-spill) window used as the beam-off / cosmic proxy
# spill starts at 206 ns (FY22) / 189 ns (FY23) -> 0-150 ns is beam-free
PREBEAM_START, PREBEAM_END = 0.0, 150.0
PREBEAM_WIDTH = PREBEAM_END - PREBEAM_START            # 150 ns

# same triggers -> trigger counts cancel; normalization is a pure livetime ratio
SPILL_WIDTH = 0.5*((spill_end_22 - spill_start_22) + (spill_end_23 - spill_start_23))  # 1531.5 ns
F_EXT = SPILL_WIDTH / PREBEAM_WIDTH
    
print('FY22:', spill_end_22 - spill_start_22)
print('FY23:', spill_end_23 - spill_start_23)
print('AVG SPILL WIDTH:', SPILL_WIDTH, ' RATIO:', F_EXT)

FY22: 1533
FY23: 1530
AVG SPILL WIDTH: 1531.5  RATIO: 10.21


In [6]:
# make MC cuts

# numuCC, nueCC, other (including nubar), NC, external
MC_PE = [[], [], [], [], []]
MC_bunch_times = [[], [], [], [], []]
MC_hits = [[], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], []]  # if weighting

inf_weights = 0     # some weights are inf for some reason

for i in trange(len(MC_cluster_time), desc = 'applying CCinc selection cuts...'):

    # No MRD cuts to avoid systematics associated with dead channels
    
    if MC_NoVeto[i] == 0:
        continue
        
    if MC_bunchTimes[i] > bunch_time_cutoff:
        continue
    
    if MC_cluster_Qb[i] > 0.2 or MC_cluster_Qb[i] < 0:
        continue
        
    if MC_cluster_number[i] != 0:
        continue
        
    brightest = MC_cluster_charge[i]; flag = False
    for j in range(i+1, len(MC_cluster_number)):
        if MC_cluster_number[j] == 0:
            break
        if MC_cluster_charge[j] > brightest:
            flag = True
            break
    if flag == True:
        continue
    
    if MC_cluster_Hits[i] < 100:
        continue

    # weight guard
    w = MC_CV_weight[i][0]
    if not np.isfinite(w):
        inf_weights += 1
        w = 1.0      # can be changed to zero
        
    # event categorization
    if MC_External[i] == True:
        indy = 4
    elif MC_NC[i] == 0:
        if MC_PDG[i] == 14:
            indy = 0
        elif MC_PDG[i] == 12:
            indy = 1
        else:
            indy = 2
    else:
        indy = 3

    MC_PE[indy].append(MC_cluster_charge[i])
    MC_hits[indy].append(MC_cluster_Hits[i])
    MC_bunch_times[indy].append(MC_bunchTimes[i])
    MC_weights_by_cat[indy].append(w)
          

print(f'{inf_weights} events had non-finite CV weights (set to 1)')
print('done')

applying CCinc selection cuts...: 100%|█| 3278461/3278461 [00:01<00:00, 2579013

14 events had non-finite CV weights (set to 1)
done


In [7]:
# data

PE = []
bunch_times = []
hits = []
reco_runs = []


for i in trange(len(cluster_time), desc = 'applying CCinc selection cuts to data...'):
    
    if(NoVeto[i]==0):   # NoVeto
        continue
    if(isBrightest[i]==0):    # Brightest
        continue
    if(cluster_Qb[i]>0.2 or cluster_Qb[i]<0): # Cluster Charge Balance < 0.2
        continue
    if(cluster_Hits[i]<100):   # indicative the muon topology is not center-aligned
        continue
        
    # Prompt (first) cluster
    if cluster_Number[i] != 0:
        continue
        
    if runs[i] > 4000:   # FY23
        if cluster_time[i] < spill_start_23:
            continue
        if cluster_time[i] > spill_end_23:
            continue
    else:
        if cluster_time[i] < spill_start_22:
            continue
        if cluster_time[i] > spill_end_22:
            continue
    
        
    # event selection
    PE.append(cluster_charge[i])
    hits.append(cluster_Hits[i])
    bunch_times.append(cluster_time[i])
    reco_runs.append(runs[i])
          

print('done')

applying CCinc selection cuts to data...: 100%|█| 3542081/3542081 [00:00<00:00,

done


In [8]:
# off-beam data

ext_PE = []
ext_bunch_times = []
ext_hits = []
ext_reco_runs = []

for i in trange(len(cluster_time), desc = 'applying CCinc selection cuts to off-beam data...'):
    
    if(NoVeto[i]==0):   # NoVeto
        continue
    if(isBrightest[i]==0):    # Brightest
        continue
    if(cluster_Qb[i]>0.2 or cluster_Qb[i]<0): # Cluster Charge Balance < 0.2
        continue
    if(cluster_Hits[i]<100):   # indicative the muon topology is not center-aligned
        continue
        
    # Prompt (first) cluster
    if cluster_Number[i] != 0:
        continue
        
    # pre-beam window instead of the spill window (no run-number split needed)
    if cluster_time[i] < PREBEAM_START or cluster_time[i] >= PREBEAM_END:
        continue
        
    # event selection
    ext_PE.append(cluster_charge[i])
    ext_hits.append(cluster_Hits[i])
    ext_bunch_times.append(cluster_time[i])
    ext_reco_runs.append(runs[i])

    
print(len(ext_PE), len(ext_bunch_times)*F_EXT)
print('done')

applying CCinc selection cuts to off-beam data...: 100%|█| 3542081/3542081 [00:

4337 44280.770000000004
done


In [11]:
# --- 1. Scaling Constants ---
POT_MC = POT/(1e20)
POT_DATA = total_POT_TOR860

#['External (MC)', r'$\nu_{\mu}$' + 'CC', r'$\nu_{e}$' + 'CC', 'NC', 'Other', 'Off-Beam Data']
# numuCC, nueCC, other (including nubar), NC, external
# --> correct ordering is [4], [0], [1], [3], [2]

ab = 0
for i in range(len(MC_bunch_times)):
    ab += len(MC_bunch_times[i])

print('MC counts:', ab)
print('On-beam counts:', len(bunch_times))
print('Off-beam counts (unscaled):', len(ext_bunch_times), ', ', len(ext_bunch_times)*F_EXT, ' (scaled)')

MC counts: 440642
On-beam counts: 214557
Off-beam counts (unscaled): 4337 ,  44280.770000000004  (scaled)


In [16]:
# Determine the correct scaling factor for the dirt events

# We do this through a simple 1D event rate scaling

# ---------------------------------------------------------
# 1D RATE NORMALIZATION (Total Yield Matching)
# ---------------------------------------------------------
f_pot = POT_DATA / POT_MC

# on-beam data, full spill window
total_data = len(bunch_times)

# pre-beam cosmics, scaled by the livetime ratio
total_off_beam = len(ext_bunch_times) * F_EXT

# non-external MC (numuCC, nueCC, other, NC), weighted
total_fixed_mc = sum(np.sum(MC_weights_by_cat[i]) * f_pot for i in [0, 1, 2, 3])
total_fixed    = total_fixed_mc + total_off_beam

# external / dirt MC, weighted and unscaled
total_ext_mc = np.sum(MC_weights_by_cat[4]) * f_pot

rate_scale_factor = (total_data - total_fixed) / total_ext_mc

# statistical uncertainty on the scale factor
var = (total_data
       + sum(np.sum(np.asarray(MC_weights_by_cat[i])**2) * f_pot**2 for i in [0, 1, 2, 3])
       + total_off_beam * F_EXT**2)
rate_scale_err = np.sqrt(var) / total_ext_mc

print("\n" + "="*45)
print(" 1D RATE NORMALIZATION")
print("="*45)
print(f"Total Data Events:         {total_data}")
print(f"Total Fixed Backgrounds:   {total_fixed:.1f}")
print(f"Total Unscaled Ext MC:     {total_ext_mc:.1f}")
print("-" * 45)
print(f"Calculated Scale Factor:   {rate_scale_factor:.3f}")
print(f"Calculated Scale Error:    {rate_scale_err:.3f}")
print("="*45 + "\n")


 1D RATE NORMALIZATION
Total Data Events:         214557
Total Fixed Backgrounds:   167472.1
Total Unscaled Ext MC:     174563.9
---------------------------------------------
Calculated Scale Factor:   0.270
Calculated Scale Error:    0.013



In [25]:
# plotting function for PMT observables

def plot_sideband_comparison(bins, mc_data, on_beam_data, off_beam_data, 
                             pot_mc, pot_data, f_ext,
                             axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                             plot_name, ext_scale_factor=1.0, legend_columns=3, 
                             mc_weights_by_cat=None, off_beam_weights=None,
                             hide_xticks=False, plot_prediction=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    
    # 2. Prepare Prediction Stack (MC + Off-Beam)
    # Order: [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE, Off-Beam Data]
    prediction_data = mc_data + [off_beam_data]

    if off_beam_weights is None:
        ext_w = np.ones(len(off_beam_data)) * f_ext
    else:
        ext_w = np.asarray(off_beam_weights) * f_ext
    
    # --- MC weights including GENIE tuning weights ---
    mc_weights = []
    for i, arr in enumerate(mc_data):
        if mc_weights_by_cat is None:
            genie_w = np.ones(len(arr))
        else:
            genie_w = np.array(mc_weights_by_cat[i])
        mc_weights.append(genie_w * f_pot)

    # Assemble full weight list
    weights = mc_weights + [ext_w]
    
    # 3. Aesthetics
    labels_base = ['External (MC)', r'$\nu_{\mu}$' + 'CC', r'$\nu_{e}$' + 'CC', 'NC', 'Other', 'Pre-Beam Data']
    colors = ['papayawhip', '#0066ff', 'cyan', 'darkred', 'darkorange', 'gray']
    # Match outline colors to the fill (or use 'navy' as in your original script)
    color_outline = ['navy'] * 5 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ext_scale_factor
            print(f"Applied {ext_scale_factor:.4f} scale to index {i} ({label})")

    # pre-scaled prediction total per bin (ext_scale_factor = 1)
    pre_weights = list(weights)
    pre_weights[0] = pre_weights[0] / ext_scale_factor if ext_scale_factor != 0 else pre_weights[0]
    pre_hist = [np.histogram(prediction_data[i], bins=bins, weights=pre_weights[i])[0]
                for i in range(len(prediction_data))]
    pre_total = np.sum(pre_hist, axis=0)


    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))
    
    # ---------------------------------------------------------
    # 5. Calculate Chi-Squared
    # ---------------------------------------------------------
    # Total variance in each bin: sigma_data^2 + sigma_mc^2
    variance = (data_err ** 2) + (pred_err ** 2)
    
    # Mask out bins where both data and MC are 0 to avoid division by zero
    valid_bins = variance > 0
    
    # Calculate chi^2 using only the valid bins
    chi2 = np.sum(((data_counts[valid_bins] - pred_counts[valid_bins]) ** 2) / variance[valid_bins])
    
    # Degrees of freedom: Number of fitted bins minus 1 (for the normalization constraint)
    ndf = np.sum(valid_bins) - 1
    
    print(f"\nExternal Scale Factor Used: {ext_scale_factor:.4f}")
    if ndf > 0:
        print(f"Chi2/NDF: {chi2:.2f} / {ndf} = {chi2 / ndf:.2f}")
    else:
        # single-bin (rate-only) plot: chi2/dof is undefined, quote the pull instead
        print(f"Single bin: pull = {np.sqrt(chi2):.2f} sigma")
    # ---------------------------------------------------------

    # --- Plotting ---
    fig = plt.figure(figsize=(5.5, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill (No edges to avoid the bar-graph look)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0)

    # B. The Outline (Reverting to your original 'step' logic for clean category boundaries)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # prediction (pre-scaled)
    if plot_prediction:
        ax.axhline(pre_total, color='red', linestyle='--', linewidth=1.2,
              label='Pre-scaled prediction')
        y_limit[1] = pre_total * 1.5

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    # Error propagation for (D/P - 1)
    # sigma_res = (D/P) * sqrt( (sigD/D)^2 + (sigP/P)^2 )
    ratio = data_counts[mask] / pred_counts[mask]
    # Handle division by zero for error calculation if data_counts is 0
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    ax.yaxis.set_minor_locator(MultipleLocator(25000))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) # Centered on 0
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    # single-bin (rate-only) plot: the x axis carries no information
    if hide_xticks:
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
        ax2.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    
    ax.legend(ncol=legend_columns, fontsize=9, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.65, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)
    
    # chi^2 stat
    if ndf > 0:
        ax.text(0.6, 0.75, r"$\chi^2 / \text{dof}$ = " + f"{chi2:.1f} / {ndf}",
                transform=ax.transAxes, fontsize = 12)

    plt.tight_layout()
    plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()


print('Plotting function loaded')

Plotting function loaded


In [26]:
# single-bin rate normalization plot (same format as the bunch timing plot)
bins = np.array([0.0, 1.0])

# put every event at x=0; the plot is rate-only, x has no meaning
mc_stack_1d = [np.zeros(len(MC_weights_by_cat[i])) for i in [4, 0, 1, 3, 2]]
mc_weights_stack_1d = [
    MC_weights_by_cat[4],
    MC_weights_by_cat[0],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[2],
]
data_1d = np.zeros(len(bunch_times))
ext_1d  = np.zeros(len(ext_bunch_times))

common_args = {
    'pot_mc': POT_MC,
    'pot_data': POT_DATA,
    'legend_columns': 2,
    'mc_weights_by_cat': mc_weights_stack_1d,
    'f_ext': F_EXT,
}

plot_sideband_comparison(
    bins=bins,
    mc_data=mc_stack_1d,
    on_beam_data=data_1d,
    off_beam_data=ext_1d,
    axis_label='',
    y_axis_label='Events',
    x_limit=[0, 1],
    y_limit=[0, 1.6 * len(bunch_times)],
    xaxis_minor=1,
    plot_name=plots_dir + '/CCinc_rate_pre-norm.png',
    ext_scale_factor=rate_scale_factor,
    plot_prediction=True,
    **common_args
)

Applied 0.2697 scale to index 0 (External (MC))

External Scale Factor Used: 0.2697
Single bin: pull = 0.00 sigma


/var/folders/5m/ny83tm4n7qq6r6gt6tbv0rg80000gq/T/ipykernel_99707/1865634246.py:166: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
